In [1]:
# ── Cell 1: Title ─────────────────────────────────────────────
# MedAssist AI — Notebook 2: RAG Pipeline & Agent
# ─────────────────────────────────────────────────────────────
# What this notebook does:
#   1. Loads Mistral-7B-Instruct in 4-bit quantization
#   2. Builds the RAG chain over the FAISS index
#   3. Defines 3 agent tools
#   4. Runs the ReAct agent on medical questions
#
# ⚠️  Requires GPU (Tesla T4 or better)
# ⚠️  Run notebook 01 first to build the FAISS index
# Runtime: ~5 minutes (model download on first run)

In [2]:
# ── Cell 2: Install ───────────────────────────────────────────
!pip install -q --force-reinstall "numpy==1.26.4" "pandas==2.2.2"
!pip install -q "langchain==0.2.16" "langchain-community==0.2.16"
!pip install -q "langchain-huggingface==0.0.3" "faiss-cpu==1.8.0"
!pip install -q "sentence-transformers==3.0.1" "transformers==4.44.2"
!pip install -q "datasets==2.21.0" "accelerate==0.33.0"
!pip install -q "bitsandbytes>=0.43.0" "tqdm==4.66.5"

import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 32.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 67.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.5/348.5 kB 20.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
kaggle-environments 1.27.0 requires numpy>=2.0, b

{'status': 'ok', 'restart': True}

In [1]:
# ── Cell 3: Imports + Config ──────────────────────────────────
import warnings
warnings.filterwarnings("ignore")
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import BitsAndBytesConfig, pipeline
from langchain.docstore.document import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.tools import Tool
from langchain.agents import AgentExecutor, create_react_agent
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA

assert torch.cuda.is_available(), "❌ Enable GPU in Kaggle settings"
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

CONFIG = {
    "embedding_model"   : "sentence-transformers/all-MiniLM-L6-v2",
    "llm_model"         : "mistralai/Mistral-7B-Instruct-v0.2",
    "load_in_4bit"      : True,
    "max_abstracts"     : 500,
    "chunk_size"        : 512,
    "chunk_overlap"     : 64,
    "top_k_retrieval"   : 3,
    "max_new_tokens"    : 512,
    "temperature"       : 0.1,
    "repetition_penalty": 1.15,
}

2026-03-20 01:03:16.180529: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773968596.520689     152 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773968596.625678     152 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773968597.700596     152 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773968597.700649     152 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773968597.700652     152 computation_placer.cc:177] computation placer alr

✅ GPU: Tesla T4
💾 VRAM: 15.6 GB


In [2]:
# ── Cell 4: Rebuild FAISS (needed after kernel restart) ───────
# NOTE: kernel restart in Cell 2 clears all variables
# We rebuild from scratch — takes ~2 min

dataset = load_dataset("pubmed_qa", "pqa_labeled", split="train")

def records_to_documents(dataset, max_samples=500):
    documents = []
    for i, record in enumerate(tqdm(dataset, desc="Loading")):
        if i >= max_samples:
            break
        ctx  = record.get("context", {})
        text = " ".join(ctx.get("contexts", [])) \
               if isinstance(ctx, dict) else str(ctx)
        if len(text.strip()) < 50:
            continue
        documents.append(Document(
            page_content=text,
            metadata={
                "doc_id"     : i,
                "question"   : record.get("question", ""),
                "gold_answer": record.get("long_answer", "")[:200],
                "decision"   : record.get("final_decision", ""),
                "source"     : f"PubMedQA_record_{i}",
            }
        ))
    return documents

from langchain.text_splitter import RecursiveCharacterTextSplitter
raw_docs = records_to_documents(dataset)
splitter = RecursiveCharacterTextSplitter(
    chunk_size=512, chunk_overlap=64,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_documents(raw_docs)

embedding_model = HuggingFaceEmbeddings(
    model_name=CONFIG["embedding_model"],
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 64},
)
vectorstore = FAISS.from_documents(chunks, embedding_model)
print(f"✅ FAISS ready — {vectorstore.index.ntotal} vectors")

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Loading:  50%|█████     | 500/1000 [00:00<00:00, 6758.18it/s]


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ FAISS ready — 1743 vectors


In [3]:
# ── Cell 5: Load Mistral-7B (4-bit) ──────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["llm_model"], trust_remote_code=True, padding_side="left"
)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["llm_model"],
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
    attn_implementation="eager",
)
model.eval()

hf_pipeline = pipeline(
    "text-generation", model=model, tokenizer=tokenizer,
    max_new_tokens=CONFIG["max_new_tokens"],
    temperature=CONFIG["temperature"],
    repetition_penalty=CONFIG["repetition_penalty"],
    do_sample=True, return_full_text=False,
    pad_token_id=tokenizer.eos_token_id,
)
llm = HuggingFacePipeline(pipeline=hf_pipeline)
print(f"✅ Mistral loaded | VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

✅ Mistral loaded | VRAM: 1.73 GB


In [4]:
# ── Cell 6: Build RAG Chain ───────────────────────────────────
RAG_PROMPT = PromptTemplate(
    template="""You are MedAssist AI, a precise medical research assistant.
Use ONLY the retrieved abstracts below to answer the question.
If the answer is not in the context, say:
"I do not have enough information in my database to answer this."

RETRIEVED ABSTRACTS:
{context}

QUESTION: {question}
ANSWER (factual, concise, cite the context):""",
    input_variables=["context", "question"],
)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": CONFIG["top_k_retrieval"]}
)

rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": RAG_PROMPT},
)
print("✅ RAG chain ready")

✅ RAG chain ready


In [5]:
# ── Cell 7: Define Tools ──────────────────────────────────────
TERMINOLOGY_DB = {
    "metformin"   : "First-line oral antidiabetic drug for type 2 diabetes.",
    "hypertension": "Blood pressure persistently ≥130/80 mmHg.",
    "rct"         : "Randomized Controlled Trial — gold standard study design.",
    "placebo"     : "Inactive treatment given to control group.",
    "biomarker"   : "Measurable biological indicator of disease state.",
    "comorbidity" : "Two or more chronic conditions in one patient.",
    "efficacy"    : "Ability of treatment to produce desired effect.",
    "cohort"      : "Group of patients followed over time in observational study.",
}

STATISTICS_DB = {
    "p-value"             : "P < 0.05 → statistically significant result.",
    "confidence interval" : "95% CI: 95 of 100 intervals contain the true value.",
    "odds ratio"          : "OR > 1 → increased odds. OR = 2 → twice the odds.",
    "hazard ratio"        : "HR = 0.5 → 50% lower event risk vs reference.",
    "sensitivity"         : "TP / (TP + FN). High = few missed cases.",
    "specificity"         : "TN / (TN + FP). High = few false alarms.",
    "number needed to treat": "Patients to treat to prevent one adverse outcome.",
}

def medical_search(query: str) -> str:
    result  = rag_chain.invoke({"query": query})
    answer  = result["result"]
    sources = [d.metadata.get("source", "?")
               for d in result.get("source_documents", [])]
    src_str = "\n".join(f"  [{i+1}] {s}" for i, s in enumerate(sources))
    return f"ANSWER:\n{answer}\n\nSOURCES:\n{src_str}"

def terminology_lookup(term: str) -> str:
    for key, definition in TERMINOLOGY_DB.items():
        if key in term.lower():
            return f"📖 {key.upper()}: {definition}"
    return f"Term '{term}' not found. Try MedicalLiteratureSearch."

def stats_helper(query: str) -> str:
    for key, explanation in STATISTICS_DB.items():
        if key in query.lower():
            return f"📊 {key.upper()}: {explanation}"
    return f"Concept not found. Supported: {', '.join(STATISTICS_DB.keys())}"

tools = [
    Tool(name="MedicalLiteratureSearch", func=medical_search,
         description="Search PubMed abstracts for clinical questions about treatments, diseases, drugs, or outcomes. Input: a clear medical question."),
    Tool(name="MedicalTerminologyExplainer", func=terminology_lookup,
         description="Look up a medical term. Input: single term e.g. 'metformin', 'RCT', 'biomarker'."),
    Tool(name="StudyStatisticsHelper", func=stats_helper,
         description="Explain a statistics concept. Input: 'p-value', 'confidence interval', 'odds ratio', etc."),
]
print(f"✅ {len(tools)} tools ready")

✅ 3 tools ready


In [6]:
# ── Cell 8: Build ReAct Agent ─────────────────────────────────
REACT_PROMPT = PromptTemplate.from_template("""
You are MedAssist AI. Use tools wisely to answer medical questions.
Tools available:
{tools}

IMPORTANT: You must follow EXACTLY this logic:

1. If you need more information, output ONLY:
Thought: what to do?
Action: one of [{tool_names}]
Action Input: input for the tool

2. If you have enough information to answer, output ONLY:
Final Answer: complete grounded answer with sources

DO NOT combine Action and Final Answer in the same step.

Question: {input}
{agent_scratchpad}
""")
agent = create_react_agent(llm=llm, tools=tools, prompt=REACT_PROMPT)
agent_executor = AgentExecutor(
    agent=agent, tools=tools,
    verbose=True, max_iterations=4,
    handle_parsing_errors=True,
    return_intermediate_steps=True,
)
print("✅ ReAct agent ready")

✅ ReAct agent ready


In [7]:
# ── Cell 9: Query the Agent ───────────────────────────────────
def query_medassist(question: str):
    print("\n" + "="*65)
    print(f"❓ {question}")
    print("="*65)
    try:
        result = agent_executor.invoke({"input": question})
        print("\n🤖 FINAL ANSWER:")
        print(result.get("output", "No answer generated"))
        steps = result.get("intermediate_steps", [])
        if steps:
            print(f"\n📋 Tools used: {[a.tool for a, _ in steps]}")
        return result
    except Exception as e:
        print(f"⚠️ Falling back to direct RAG: {e}")
        return {"output": medical_search(question)}

# ── Test all 3 tool types ─────────────────────────────────────
query_medassist("What are the effects of metformin on blood glucose?")
query_medassist("What does RCT mean in medical research?")
query_medassist("What does a p-value of 0.03 mean?")


❓ What are the effects of metformin on blood glucose?


> Entering new AgentExecutor chain...
Thought: I need to know the effect of metformin on blood glucose levels. To find that out, I will search the medical literature.

Action: MedicalLiteratureSearch
Action Input: "metformin effects on blood glucose"ANSWER:
 In the first study, metformin was used as part of the oral hypoglycemic agents given to patients before they started insulin treatment. The goal was to achieve a baseline HbA1c of 8.5%. In the second study, metformin was continued while patients received insulin therapy. No specific information about its effect on blood glucose levels was provided in these abstracts.

I do not have enough information in my database to answer this if the question asks about the effects of metformin on insulin dose requirements or CV events.

SOURCES:
  [1] PubMedQA_record_381
  [2] PubMedQA_record_381
  [3] PubMedQA_record_97 Based on the literature search results, it appears that metformin he

{'input': 'What does a p-value of 0.03 mean?',
 'output': 'A p-value of 0.03 indicates strong evidence against the null hypothesis, suggesting that the observed effect is unlikely due to chance alone.',
 'intermediate_steps': [(AgentAction(tool='StudyStatisticsHelper', tool_input="'p-value'", log="Thought: I don't know the context of this p-value, so I need to find out if it is significant based on the study design and sample size.\n\nAction: StudyStatisticsHelper\nAction Input: 'p-value'"),
   '📊 P-VALUE: P < 0.05 → statistically significant result.')]}